# AEM4L5 · Notebook 4 — Caso integrador (end-to-end)

**Clase:** Arquitecturas avanzadas de adaptación.

Integramos LoRA/PEFT, serving, profiling y async en una sola arquitectura.

## 0. Setup tecnico

Instalamos solo `langchain-core` (no requiere API key). `langchain-openai` es **opcional**.

Toda la clase corre **sin claves**: simulamos el modelo con `RunnableLambda`.

In [ ]:
# Dependencia base (no requiere API key):
!pip install -q langchain-core

# Opcional, solo si quisieras usar OpenAI real (no es necesario para esta clase):
# !pip install -q langchain-openai

In [ ]:
import time
import asyncio
import cProfile
import pstats
import io
import re
from typing import List, Dict, Any

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel


def fake_llm_response(prompt_value):
    # Simulador de modelo: evita depender de APIs pagas.
    text = prompt_value.to_string() if hasattr(prompt_value, "to_string") else str(prompt_value)
    return f"Respuesta simulada del modelo para:\n{text[:300]}"


fake_llm = RunnableLambda(fake_llm_response)
print("Setup listo. Modelo simulado disponible como fake_llm.")

## 1. Objetivo del notebook

Diseñar una **arquitectura completa** para una startup que necesita resumir documentos legales con **bajo costo** y **tráfico irregular**, combinando todo lo visto en N1–N3.

## 2. El caso

Una startup quiere ofrecer resúmenes personalizados de documentos legales.

**Requisitos:**
- tráfico irregular;
- bajo presupuesto;
- personalización por dominio o cliente;
- documentos en cloud storage;
- latencia promedio objetivo < 1.5 s;
- necesidad de medir y optimizar el pipeline;
- posibilidad de crecer a tráfico sostenido.

## 3. Mapa visual de la arquitectura

```mermaid
flowchart LR
    A[Usuario] --> B[API]
    B --> C[LangChain Router]
    C --> D{Dominio}
    D --> E[Adaptador legal]
    D --> F[Adaptador salud]
    D --> G[Adaptador soporte]
    E --> H[Modelo base]
    F --> H
    G --> H
    H --> I[Resumen]
    I --> J[Postprocesamiento]
    J --> K[Respuesta]
    B --> L[Profiling]
    B --> M[Metricas]
    B --> N[Async I/O]
```

## 4. Glosario (repaso integrador)

| Término | En este caso |
|---|---|
| LoRA / PEFT | Un adaptador liviano por dominio/cliente sobre un modelo base compartido. |
| Serverless | Despliegue inicial barato para tráfico irregular. |
| Cold start | Riesgo a vigilar contra el objetivo de < 1.5 s. |
| cProfile | Para hallar hotspots de CPU (pre/postprocesamiento). |
| async / abatch | Para solapar I/O: leer storage, llamar al LLM. |
| Métricas | Latencia, costo por request, tasa de error, calidad. |

## 5. Teoría: cómo se conectan las decisiones

Cada decisión de N1–N3 encaja en el caso:

1. **Adaptación (N1):** modelo base + adaptadores LoRA por dominio → barato y con muchas variantes.
2. **Despliegue (N2):** empezar **serverless** (tráfico irregular, presupuesto bajo); migrar a servidor persistente si el tráfico se vuelve constante.
3. **Rendimiento (N3):** `cProfile` para el pre/postprocesamiento (CPU); `async`/`abatch` para storage y LLM (I/O).
4. **Observabilidad:** medir latencia, costo y calidad desde el día uno.

## 6. Tabla de decisiones de arquitectura

| Decisión | Recomendación |
|---|---|
| Adaptación | Modelo base + adaptadores LoRA por dominio o cliente |
| Orquestación | LangChain para prompts, routing y chains |
| Deploy inicial | Serverless si la latencia tolera cold start |
| Deploy futuro | Servidor persistente si el tráfico se vuelve constante |
| Diagnóstico | cProfile para CPU-bound |
| I/O | asyncio / abatch para llamadas externas |
| Métricas | latencia, costo por request, tasa de error, calidad de resumen |
| Riesgo | cold start, mala elección de adaptador, falta de observabilidad |

## 7. Gráfico Mermaid: routing por dominio (LangChain)

```mermaid
flowchart TB
    A[Documento + dominio] --> B[LangChain Router]
    B -->|legal| C[Adapter legal]
    B -->|salud| D[Adapter salud]
    B -->|soporte| E[Adapter soporte]
    C --> F[Modelo base compartido]
    D --> F
    E --> F
    F --> G[Resumen adaptado]
```

## 8. Mini ejemplo conceptual: router de dominio simulado

Simulamos el router que elige el adaptador según el dominio del documento.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

prompt = ChatPromptTemplate.from_template(
    "Resumi este documento de dominio {dominio}:\n\n{documento}"
)

def router_adaptadores(prompt_value):
    text = prompt_value.to_string().lower()
    if "legal" in text:
        return "[adapter legal] Resumen juridico simulado."
    if "salud" in text:
        return "[adapter salud] Resumen clinico simulado."
    return "[adapter soporte] Resumen de soporte simulado."

router_chain = prompt | RunnableLambda(router_adaptadores)

for d in ["legal", "salud", "soporte"]:
    print(router_chain.invoke({"dominio": d, "documento": "Documento de ejemplo..."}))

## 9. Código Python + LangChain: pipeline async del caso

Procesamos varios documentos en paralelo lógico con `abatch` (I/O-bound: simula leer storage + llamar al LLM).

In [ ]:
import asyncio, time
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

prompt = ChatPromptTemplate.from_template(
    "Resumi en {lineas} lineas el documento {doc_id}:\n\n{documento}"
)

async def fake_resumen(prompt_value):
    await asyncio.sleep(0.5)  # simula I/O: storage + LLM
    text = prompt_value.to_string()
    return f"Resumen simulado ({len(text)} chars de prompt)."

pipeline = prompt | RunnableLambda(fake_resumen)

docs = [{"lineas": 3, "doc_id": i, "documento": f"Contrato {i}. " * 80} for i in range(6)]

inicio = time.time()
resultados = await pipeline.abatch(docs)
fin = time.time()
print(f"Procesados {len(resultados)} docs en {round(fin - inicio, 2)} s (async).")
print(resultados[0])

## 10. Ejercicio guiado — Documento de arquitectura

Completá esta plantilla para el caso (producto, restricciones, adaptación, despliegue, LangChain, profiling, async, métricas, riesgos, migración).

In [ ]:
documento_arquitectura = """
# Documento de arquitectura

## Producto
Servicio de resumen legal personalizado.

## Restricciones
Trafico irregular, presupuesto bajo, documentos en cloud storage, latencia objetivo < 1.5 s.

## Estrategia de adaptacion
Modelo base compartido con adaptadores LoRA por dominio o cliente.

## Patron de despliegue
Empezar serverless para reducir costo ocioso; evaluar servidor persistente si el trafico se vuelve constante.

## Uso de LangChain
Orquesta prompts, routing por dominio, chains de resumen y conexion con el modelo adaptado.

## Estrategia de profiling
cProfile para preprocesamiento, postprocesamiento y funciones auxiliares.

## Estrategia async
async/abatch para lectura de documentos, llamadas al proveedor LLM o base remota.

## Metricas
Latencia promedio, p95, costo por request, tasa de error, calidad de resumen, cold start.

## Riesgos
Cold start, adaptador incorrecto, sobrecarga de concurrencia, falta de evaluacion.

## Migracion futura
Migrar a servidor persistente si el trafico pasa a ser constante o la latencia se vuelve estricta.
"""
print(documento_arquitectura)

## 11. Preguntas de interpretación

1. ¿Por qué serverless **al inicio** y no servidor persistente?
2. ¿Qué parte del pipeline es CPU-bound y cuál I/O-bound?
3. ¿Qué métrica te avisaría que conviene **migrar** de patrón?
4. ¿Qué pasa si el router elige el **adaptador equivocado**?

## 12. Errores comunes y cierre

**Errores comunes:** elegir el patrón de deploy sin medir tráfico; no observar métricas; usar async para cálculo puro; no versionar adaptadores; optimizar antes de perfilar.

**Cierre:** una demo de IA pregunta *“¿el modelo responde?”*. Un **producto** de IA pregunta *“¿responde bien, rápido, barato, medible y bajo carga?”*. Esta clase te dio el marco para responder esa segunda pregunta: **adaptar** (LoRA/PEFT), **servir** (serverless/servidor), **medir** (cProfile) y **escalar la espera** (async).